# MPOPI + PPO trên mjlab (Google Colab, GPU)

Notebook này chạy nhánh `mpopi-ppo` của mjlab trên GPU của Colab:

1. Cài đặt môi trường (`uv`, PyTorch CUDA, MuJoCo Warp).
2. Chạy unit test của MPOPI.
3. Chạy thử nhanh (smoke test) trên GPU.
4. Benchmark A/B/C trên `Mjlab-Cartpole-Balance`, vẽ đồ thị và kiểm định thống kê.
5. (Tùy chọn) Train robot thật, ví dụ Unitree G1, với 3 chế độ: `ppo`, `naive_replay_ppo`, `mpopi_ppo`.
6. Theo dõi log và TensorBoard trong lúc train, xem video policy, tải kết quả về.

**Trước khi chạy:** chọn *Runtime → Change runtime type → GPU*. T4 đủ cho Cartpole. Với G1 nên dùng L4 hoặc A100.

> **Lưu ý:** code MPOPI mới chỉ được kiểm thử trên CPU. Đây là lần đầu chạy trên GPU, nên hãy chạy mục 3 (smoke test) trước khi chạy các thí nghiệm dài.

## 0. Cấu hình

In [ ]:
#@title Cấu hình chung
#@markdown **Nguồn code:** `github` clone nhánh từ GitHub (phải push nhánh trước); `zip` upload file zip của repo.
SOURCE = "github"  #@param ["github", "zip"]
REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"  #@param {type:"string"}
BRANCH = "mpopi-ppo"  #@param {type:"string"}
#@markdown **Lưu kết quả lên Google Drive** (nên bật, vì Colab có thể ngắt kết nối và mất dữ liệu):
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

import os

REPO_DIR = "/content/mjlab_MPOPI"
if SAVE_TO_DRIVE:
  from google.colab import drive

  drive.mount("/content/drive")
  OUT_ROOT = "/content/drive/MyDrive/mpopi_results"
else:
  OUT_ROOT = "/content/mpopi_results"
os.makedirs(OUT_ROOT, exist_ok=True)
print("Kết quả sẽ lưu tại:", OUT_ROOT)

In [ ]:
#@title Hàm tiện ích: chạy lệnh và in log trực tiếp
import os
import subprocess
import sys


def run(cmd: str, cwd: str | None = None) -> None:
  """Run a shell command, stream its output, and raise if it fails."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd,
    shell=True,
    cwd=cwd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ},
  )
  assert proc.stdout is not None
  for line in proc.stdout:
    print(line, end="", flush=True)
  if proc.wait() != 0:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy code

- `SOURCE = "github"`: clone nhánh `BRANCH`. Nhánh `mpopi-ppo` phải được push lên GitHub trước. Nếu repo private, dùng URL có token hoặc chọn cách zip.
- `SOURCE = "zip"`: upload file `mjlab_MPOPI_mpopi-ppo.zip` (tạo bằng `git archive`, xem README ở mục cuối).

In [ ]:
import os
import shutil
import zipfile

if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)

if SOURCE == "github":
  run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
else:
  from google.colab import files

  uploaded = files.upload()  # Chọn file mjlab_MPOPI_mpopi-ppo.zip
  name = next(iter(uploaded))
  with zipfile.ZipFile(name) as zf:
    zf.extractall("/content")
  assert os.path.isdir(REPO_DIR), f"Không thấy {REPO_DIR} sau khi giải nén"

# Kiểm tra code đủ mới: nhánh trên GitHub có thể chưa được push bản mới nhất.
REQUIRED = [
  "src/mjlab/rl/mpopi/algorithm.py",
  "tests/test_mpopi_algorithm.py",
  "scripts/benchmarks/mpopi_benchmark.py",
]
missing = [f for f in REQUIRED if not os.path.exists(f"{REPO_DIR}/{f}")]
if missing:
  raise RuntimeError(
    "Code đã cũ, thiếu: " + ", ".join(missing) + ". "
    "Hãy push bản mới nhất của nhánh lên GitHub, hoặc dùng SOURCE = 'zip'."
  )
if SOURCE == "github":
  run("git log --oneline -1", cwd=REPO_DIR)
print("OK: code có đủ các file MPOPI cần thiết.")

## 3. Cài đặt môi trường

Cài `uv` rồi `uv sync` (PyTorch bản CUDA, MuJoCo Warp, RSL-RL 5.5.1). Lần đầu mất khoảng 5–10 phút.

Trên Colab dùng `uv sync` **không** kèm `--extra cpu` để có PyTorch bản CUDA.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
run("uv sync", cwd=REPO_DIR)
run(
  "uv run python -c \"import torch, warp, mujoco_warp, rsl_rl; "
  "print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), "
  "torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')\"",
  cwd=REPO_DIR,
)

## 4. Unit test của MPOPI

In [ ]:
run(
  "uv run pytest tests/test_mpopi_estimators.py tests/test_mpopi_replay.py "
  "tests/test_mpopi_algorithm.py -q -p no:warnings",
  cwd=REPO_DIR,
)

## 5. Smoke test trên GPU

Train Cartpole 5 vòng lặp ở chế độ `mpopi_ppo` bằng CLI `train` thật. Nếu bước này lỗi, đừng chạy các bước sau.

In [ ]:
run(
  "uv run train Mjlab-Cartpole-Balance --agent.logger tensorboard "
  "--env.scene.num-envs 512 --agent.max-iterations 5 "
  "--agent.algorithm.mpopi.mode mpopi_ppo --agent.algorithm.mpopi.replay-buffer-size 2 "
  f"--log-root {OUT_ROOT}/smoke",
  cwd=REPO_DIR,
)

## 6. Benchmark A/B/C trên Cartpole-Balance

Các nhánh so sánh:

| Nhánh | Ý nghĩa |
|---|---|
| `A_ppo` | PPO chuẩn |
| `A_ppo_bigmb` | PPO với số mẫu mỗi bước gradient bằng các nhánh replay (đối chứng cân bằng tính toán) |
| `B_naive_replay` | Replay dữ liệu cũ, coi như on-policy, không hiệu chỉnh |
| `C_mpopi` | Replay có hiệu chỉnh importance sampling (ρ̄ = 1) |
| `C_mpopi_noclip` | Như C nhưng không chặn trọng số |

Điểm số là reward trung bình mỗi bước của policy tất định trên env `play` riêng. Giá trị tối đa là 0,05 (1.0 sau chuẩn hóa).

Kết quả trên CPU (5 seed, xem `docs/mpopi_cartpole_results.md`) chưa đủ để kết luận. Trên GPU nên chạy **≥ 10 seed**. Nên chốt giả thuyết trước khi chạy.

**Log trong lúc chạy.** Cứ mỗi `PROGRESS_EVERY` vòng lặp, script in một dòng tiến độ, ví dụ:

```
[C_mpopi seed 300] it   41/80  eval    0.0493  train_r   0.0412  kl 0.014  clip 0.171  ess 0.957  w 0.893  kl_mu 0.068
```

| Cột | Ý nghĩa |
|---|---|
| `it 41/80` | Vòng lặp hiện tại / tổng số vòng |
| `eval` | Điểm của policy tất định ở lần đánh giá gần nhất (tối đa 0,05 với Cartpole) |
| `train_r` | Reward trung bình mỗi bước khi train (có nhiễu khám phá, nên thấp hơn `eval`) |
| `kl`, `clip` | KL giữa policy trước và sau update; tỉ lệ mẫu bị PPO clip (chỉ có ở nhánh replay) |
| `ess` | Số mẫu hiệu dụng của trọng số replay (1 = replay gần như on-policy) |
| `w` | Trọng số importance trung bình của replay |
| `kl_mu` | KL giữa policy cũ đã sinh replay (μ) và policy hiện tại: dữ liệu càng cũ thì càng lớn |

Cuối mỗi lần chạy có một dòng `final score`.

In [ ]:
#@title Tham số benchmark
TASK = "Mjlab-Cartpole-Balance"  #@param {type:"string"}
NUM_ENVS = 256  #@param {type:"integer"}
SEEDS = 10  #@param {type:"integer"}
SEED_OFFSET = 300  #@param {type:"integer"}
ITERATIONS = 80  #@param {type:"integer"}
EVAL_EVERY = 5  #@param {type:"integer"}
ARMS = "A_ppo A_ppo_bigmb B_naive_replay C_mpopi C_mpopi_noclip"  #@param {type:"string"}
REPLAY_BUFFER_SIZE = 4  #@param {type:"integer"}
REPLAY_RATIO = 1.0  #@param {type:"number"}
PROGRESS_EVERY = 10  #@param {type:"integer"}

BENCH_DIR = f"{OUT_ROOT}/bench_{TASK}_{NUM_ENVS}envs"

In [ ]:
run(
  "uv run python scripts/benchmarks/mpopi_benchmark.py "
  f"--task {TASK} --device cuda:0 --num-envs {NUM_ENVS} "
  f"--seeds {SEEDS} --seed-offset {SEED_OFFSET} --iterations {ITERATIONS} "
  f"--eval-every {EVAL_EVERY} --arms {ARMS} "
  f"--replay-buffer-size {REPLAY_BUFFER_SIZE} --replay-ratio {REPLAY_RATIO} "
  f"--progress-every {PROGRESS_EVERY} --out-dir {BENCH_DIR}",
  cwd=REPO_DIR,
)

### 6.1 Đồ thị và kiểm định theo cặp seed

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

MAX_REWARD = 0.05  # reward weight 1.0 x dt 0.05 cho Cartpole; đổi nếu dùng task khác.

df = pd.read_csv(f"{BENCH_DIR}/curves.csv")
ev = df[df["eval_return"].notna()].copy()
ev["score"] = ev["eval_return"] / MAX_REWARD

fig, ax = plt.subplots(figsize=(8, 4.5))
for arm, g in ev.groupby("arm"):
  stat = g.groupby("env_steps")["score"].agg(["mean", "std", "count"])
  ci = 1.96 * stat["std"] / stat["count"].pow(0.5)
  ax.plot(stat.index, stat["mean"], label=arm)
  ax.fill_between(stat.index, stat["mean"] - ci, stat["mean"] + ci, alpha=0.15)
ax.set_xlabel("Số bước môi trường")
ax.set_ylabel("Điểm chuẩn hóa (1.0 = tối đa)")
ax.set_title(f"{TASK}, {NUM_ENVS} envs, {SEEDS} seeds (mean ± 95% CI)")
ax.grid(alpha=0.3)
ax.legend()
fig.savefig(f"{BENCH_DIR}/curves.png", dpi=150, bbox_inches="tight")
plt.show()

summary = json.load(open(f"{BENCH_DIR}/summary.json"))
arms = [a for a in ARMS.split() if a in summary]
table = pd.DataFrame(
  {
    a: {
      "AUC": summary[a]["auc"]["mean"] / MAX_REWARD,
      "AUC ±95%": summary[a]["auc"]["ci95"] / MAX_REWARD,
      "final": summary[a]["final"]["mean"] / MAX_REWARD,
      "final ±95%": summary[a]["final"]["ci95"] / MAX_REWARD,
    }
    for a in arms
  }
).T
display(table.round(3))

# Kiểm định theo cặp: cùng seed = cùng khởi tạo mạng và môi trường.
rows = []
for a, b in [
  ("C_mpopi", "A_ppo"),
  ("C_mpopi", "A_ppo_bigmb"),
  ("C_mpopi", "B_naive_replay"),
  ("B_naive_replay", "A_ppo"),
  ("C_mpopi_noclip", "C_mpopi"),
]:
  if a not in summary or b not in summary:
    continue
  d = [
    (x - y) / MAX_REWARD
    for x, y in zip(summary[a]["per_seed"]["auc"], summary[b]["per_seed"]["auc"])
  ]
  p = stats.wilcoxon(d).pvalue if any(v != 0 for v in d) else math.nan
  rows.append(
    {
      "so sánh (AUC)": f"{a} − {b}",
      "chênh lệch TB": sum(d) / len(d),
      "số seed âm": f"{sum(v < 0 for v in d)}/{len(d)}",
      "Wilcoxon p": p,
    }
  )
display(pd.DataFrame(rows).round(4))

## 7. (Tùy chọn) Train robot với 3 chế độ

Chạy CLI `train` lần lượt cho từng chế độ với cùng seed. Mặc định là `Mjlab-Velocity-Flat-Unitree-G1`.

**Train chạy nền.** Cell dưới đây khởi động train rồi trả về ngay, để bạn vẫn chạy được các cell khác trong lúc train:
- mục 7.1 xem tiến độ (chạy lại cell bất cứ lúc nào để cập nhật);
- mục 8 TensorBoard (tự cập nhật trong lúc train);
- mục 10 xem video của checkpoint mới nhất.

Log đầy đủ của từng chế độ được ghi vào `train_console/<chế độ>_seed<seed>.log`.

Lưu ý:
- Mỗi lần train G1 có thể mất hàng giờ. Colab có thể ngắt khi tab bị bỏ không quá lâu, nên bật `SAVE_TO_DRIVE` ở mục 0.
- Replay buffer không được lưu vào checkpoint: nếu resume, buffer bắt đầu lại từ rỗng.
- Log bằng TensorBoard để khỏi phải đăng nhập wandb.

In [ ]:
#@title Tham số train
ROBOT_TASK = "Mjlab-Velocity-Flat-Unitree-G1"  #@param {type:"string"}
ROBOT_NUM_ENVS = 2048  #@param {type:"integer"}
ROBOT_ITERATIONS = 500  #@param {type:"integer"}
ROBOT_SEED = 1  #@param {type:"integer"}
MODES = "ppo naive_replay_ppo mpopi_ppo"  #@param {type:"string"}

ROBOT_LOG_ROOT = f"{OUT_ROOT}/train_logs"

In [ ]:
import shlex
import subprocess

CONSOLE_DIR = f"{OUT_ROOT}/train_console"
os.makedirs(CONSOLE_DIR, exist_ok=True)

if "TRAIN_PROC" in globals() and TRAIN_PROC.poll() is None:
  raise RuntimeError("Đang có một lần train chạy nền. Dừng nó ở mục 7.2 trước.")

commands = []
for mode in MODES.split():
  log_file = shlex.quote(f"{CONSOLE_DIR}/{mode}_seed{ROBOT_SEED}.log")
  commands.append(
    f"uv run train {ROBOT_TASK} --agent.logger tensorboard "
    f"--env.scene.num-envs {ROBOT_NUM_ENVS} --agent.max-iterations {ROBOT_ITERATIONS} "
    f"--agent.seed {ROBOT_SEED} --agent.run-name {mode}_seed{ROBOT_SEED} "
    f"--agent.algorithm.mpopi.mode {mode} --log-root {ROBOT_LOG_ROOT} "
    f"> {log_file} 2>&1"
  )
# Các chế độ chạy lần lượt; nếu một chế độ lỗi thì dừng luôn.
TRAIN_PROC = subprocess.Popen(
  ["bash", "-c", " && ".join(commands)],
  cwd=REPO_DIR,
  env={**os.environ},
  start_new_session=True,
)
print(f"Đã bắt đầu train nền (PID {TRAIN_PROC.pid}): {MODES}")
print("Xem tiến độ ở mục 7.1, TensorBoard ở mục 8.")

### 7.1 Xem tiến độ train

Chạy lại cell này bất cứ lúc nào. Nó in khối log mới nhất của từng chế độ và vẽ `Mean reward` theo vòng lặp.

- `Mean reward` chỉ xuất hiện sau khi có episode đầu tiên kết thúc.
- Đặt `FULL_LOG = True` để xem toàn bộ khối log, gồm từng thành phần reward (`Episode_Reward/...`).

In [ ]:
#@title Xem tiến độ
FULL_LOG = False  #@param {type:"boolean"}
import glob
import re

import matplotlib.pyplot as plt

ANSI = re.compile(r"\x1b\[[0-9;]*m")
KEY_LINES = re.compile(
  r"Learning iteration|Total steps|Mean reward|Mean episode length|"
  r"Mean (surrogate|value|entropy|kl|clip_fraction) loss|"
  r"mpopi/(ess|weight_mean|accepted|behavior_kl|policy_age_mean)|"
  r"Mean action std|Iteration time|ETA"
)


def parse_log(path: str) -> tuple[list[tuple[int, float]], str]:
  """Return (iteration, mean reward) pairs and the latest log block."""
  text = ANSI.sub("", open(path, errors="ignore").read())
  blocks = text.split("Learning iteration")[1:]
  curve = []
  for block in blocks:
    it = re.match(r"\s*(\d+)/", block)
    rew = re.search(r"Mean reward:\s*(-?[\d.]+)", block)
    if it and rew:
      curve.append((int(it.group(1)), float(rew.group(1))))
  latest = "Learning iteration" + blocks[-1] if blocks else ""
  return curve, latest


logs = sorted(glob.glob(f"{OUT_ROOT}/train_console/*.log"), key=os.path.getmtime)
if not logs:
  print("Chưa có log. Hãy chạy mục 7 trước.")
curves = {}
for path in logs:
  name = os.path.basename(path).removesuffix(".log")
  curve, latest = parse_log(path)
  curves[name] = curve
  print(f"===== {name}")
  if not latest:
    tail = open(path, errors="ignore").read().splitlines()[-5:]
    print("Đang khởi tạo... (5 dòng cuối của log)")
    print("\n".join(ANSI.sub("", t) for t in tail))
    continue
  lines = latest.splitlines()
  shown = lines if FULL_LOG else [ln for ln in lines if KEY_LINES.search(ln)]
  print("\n".join(ln.strip() for ln in shown if ln.strip()))

running = "TRAIN_PROC" in globals() and TRAIN_PROC.poll() is None
print("\nTrạng thái:", "ĐANG TRAIN" if running else "không có train nào đang chạy")

if any(curves.values()):
  fig, ax = plt.subplots(figsize=(8, 4))
  for name, curve in curves.items():
    if curve:
      xs, ys = zip(*curve)
      ax.plot(xs, ys, label=name)
  ax.set_xlabel("Vòng lặp")
  ax.set_ylabel("Mean reward (episode)")
  ax.grid(alpha=0.3)
  ax.legend()
  plt.show()

### 7.2 Chờ train xong hoặc dừng train

- Cell thứ nhất chờ tới khi train xong và in tiến độ mỗi 5 phút. Bấm *Stop* để ngừng **chờ**; train vẫn chạy nền.
- Cell thứ hai **dừng hẳn** train.

In [ ]:
import time

if "parse_log" not in globals():
  raise RuntimeError("Hãy chạy cell 7.1 (Xem tiến độ) một lần trước.")
while "TRAIN_PROC" in globals() and TRAIN_PROC.poll() is None:
  time.sleep(300)
  logs = sorted(glob.glob(f"{OUT_ROOT}/train_console/*.log"), key=os.path.getmtime)
  if logs:
    _, latest = parse_log(logs[-1])
    head = [ln.strip() for ln in latest.splitlines() if KEY_LINES.search(ln)][:4]
    print(time.strftime("%H:%M"), os.path.basename(logs[-1]), "|", " | ".join(head))
print("Train đã kết thúc, mã thoát:", TRAIN_PROC.returncode if "TRAIN_PROC" in globals() else None)

In [ ]:
import signal

if "TRAIN_PROC" in globals() and TRAIN_PROC.poll() is None:
  os.killpg(TRAIN_PROC.pid, signal.SIGTERM)
  print("Đã gửi lệnh dừng train.")
else:
  print("Không có train nào đang chạy.")

## 8. TensorBoard

Mở được **trong lúc train** (mục 7 chạy nền); đồ thị tự cập nhật mỗi 30 giây.

Các chỉ số cần xem:
- `Train/mean_reward`, `Train/mean_episode_length`.
- `Loss/kl`, `Loss/clip_fraction`: chỉ có ở các chế độ replay.
- `Loss/mpopi/ess`, `Loss/mpopi/weight_mean`, `Loss/mpopi/clipped_frac`, `Loss/mpopi/behavior_kl`, `Loss/mpopi/accepted`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUT_ROOT} --reload_interval 30

## 9. Tải kết quả về máy

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/mpopi_results", "zip", OUT_ROOT)
files.download(archive)

## 10. Xem policy đã train (video)

Colab không có màn hình, nên `play` quay một video từ checkpoint rồi hiển thị ngay trong notebook.

- Dùng được **cả trong lúc train**: checkpoint được lưu mỗi 50 vòng lặp.
- Checkpoint của cả ba chế độ đều mở được như nhau: `play` chỉ nạp mạng actor, và mạng này giống hệt PPO thường.
- Mặc định chọn checkpoint mới nhất trong `CKPT_GLOB`; có thể gán `CKPT` để chọn tay.

In [ ]:
#@title Quay video từ checkpoint
PLAY_TASK = "Mjlab-Velocity-Flat-Unitree-G1"  #@param {type:"string"}
CKPT_GLOB = "train_logs/**/model_*.pt"  #@param {type:"string"}
CKPT = ""  #@param {type:"string"}
VIDEO_STEPS = 400  #@param {type:"integer"}
import glob
import signal
import subprocess
import time

from IPython.display import Video

if not CKPT:
  found = glob.glob(f"{OUT_ROOT}/{CKPT_GLOB}", recursive=True)
  assert found, f"Không tìm thấy checkpoint theo {OUT_ROOT}/{CKPT_GLOB}"
  CKPT = max(found, key=os.path.getmtime)
print("Checkpoint:", CKPT)

video = os.path.join(os.path.dirname(CKPT), "videos/play/rl-video-step-0.mp4")
if os.path.exists(video):
  os.remove(video)
# Sau khi quay xong, play vẫn chạy viewer web, nên chạy nền rồi tắt khi có video.
proc = subprocess.Popen(
  f"uv run play {PLAY_TASK} --checkpoint-file {CKPT} --video True "
  f"--video-length {VIDEO_STEPS} --viewer viser",
  shell=True,
  cwd=REPO_DIR,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.STDOUT,
  start_new_session=True,
)
start, last_size = time.time(), -1
while time.time() - start < 900:
  time.sleep(5)
  if os.path.exists(video):
    size = os.path.getsize(video)
    if size == last_size and size > 0:
      break
    last_size = size
os.killpg(proc.pid, signal.SIGTERM)
assert os.path.exists(video), "Không tạo được video. Thử chạy lệnh play trực tiếp để xem lỗi."
Video(video, embed=True, width=640)

## Ghi chú

**Tạo file zip của nhánh** (trên máy local, trong thư mục repo), khi không muốn push lên GitHub:

```bash
git archive --format=zip --prefix=mjlab_MPOPI/ -o mjlab_MPOPI_mpopi-ppo.zip mpopi-ppo
```

**Ý nghĩa các chế độ `--agent.algorithm.mpopi.mode`:**
- `ppo`: PPO chuẩn của RSL-RL, không thay đổi gì.
- `naive_replay_ppo`: thêm dữ liệu cũ vào batch PPO như thể on-policy.
- `mpopi_ppo`: thêm dữ liệu cũ, có hiệu chỉnh importance sampling (trọng số `clip(π_old/μ)`, V-trace).